In [81]:
from mhcicleave.cleavagedatasets import PeptideDataset
from mhcicleave.encodings import Embedder
import torch
from pathlib import Path
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


### Pandas sampling gives different results than huggingface dataset shuffling and select.

In [93]:

def downsample_data_pd(dataset, label_col, percentage, balance_classes, seed=None):
    """
    Downsamples a pandas DataFrame, with options to balance classes or 
    take a percentage of each class, replicating the datasets logic.

    Args:
        dataset (pd.DataFrame): The input DataFrame.
        label_col (str): The name of the column to use as the label.
        percentage (float): The percentage (0-100) of each class to keep.
                            Only used if balance_classes is False.
        balance_classes (bool): If True, downsamples the negative class
                                to match the number of positive samples.
        seed (int, optional): The random seed for shuffling and sampling
                              to ensure reproducibility. Defaults to None.

    Returns:
        pd.DataFrame: The downsampled and shuffled DataFrame.
    """
    if label_col not in dataset.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    # Split dataset into positive (1) and negative (not 1)
    pos = dataset[dataset[label_col] == 1]
    neg = dataset[dataset[label_col] != 1]
    n_pos, n_neg = len(pos), len(neg)

    if balance_classes:
        logging.info(f"Balancing classes using {n_pos} positive samples.")
        if n_pos == 0 or n_neg == 0:
             logging.warning("Cannot balance with zero samples in one class.")
             # Return an empty DataFrame with the same columns
             return dataset.iloc[0:0] 
        
        # Original logic: sample negative class to match positive
        n_neg_target = min(n_pos, n_neg)
        
        # .sample(n=...) is the pandas equivalent of .shuffle().select(range(n))
        sampled_neg = neg.sample(n=n_neg_target, random_state=seed)
        
        # pd.concat is the equivalent of concatenate_datasets
        final_dataset = pd.concat([pos, sampled_neg])
        
        # .sample(frac=1) is the equivalent of .shuffle()
        final_dataset = final_dataset.sample(
            frac=1, random_state=seed
        ).reset_index(drop=True)
        
        logging.info(f"Created balanced dataset with {len(final_dataset)} samples.")
    
    else: # Percentage per class
        if not (0 < percentage <= 100):
            raise ValueError('Percentage must be > 0 and <= 100.')
        if percentage == 100:
            return dataset

        n_pos_target = int(n_pos * (percentage / 100.0))
        print(n_pos_target)
        n_neg_target = int(n_neg * (percentage / 100.0))
        print(n_neg_target)
        # Use .sample() to get n_pos_target random samples
        sampled_pos = pos.sample(n=n_pos_target, random_state=seed) if n_pos > 0 else pos
        sampled_neg = neg.sample(n=n_neg_target, random_state=seed) if n_neg > 0 else neg
        
        # Concatenate the two sampled sets
        final_dataset = pd.concat([sampled_pos, sampled_neg]).reset_index(drop=True)
        
        # Shuffle the final combined dataset
        final_dataset = final_dataset.sample(frac=1, random_state=seed).reset_index(drop=True)
        
        logging.info(f"Created {percentage}% per-class dataset with {len(final_dataset)} samples.")
    
    return final_dataset

In [94]:
seed = 42
balance_classes = False
downsample_percentage = 10.
label_col = 'elution_cleavage'

In [95]:
train_data = '../data/raw/c_train.csv'
dataset_pd = pd.read_csv(train_data)
processed_dataset = downsample_data_pd(dataset_pd, label_col, downsample_percentage, balance_classes, seed)

2025-10-17 17:26:59,033 - INFO - Created 10.0% per-class dataset with 113559 samples.


20088
93471


In [96]:
ds_suffix = "balanced" if balance_classes else (f"{int(downsample_percentage)}pct_per_class" if downsample_percentage != 100.0 else "full")
out_fname = f'{Path(train_data).stem}_{len(processed_dataset[label_col])}samples_{ds_suffix}.csv'
processed_dataset.to_csv(f'../data/processed/{out_fname}',index=None)

## Embedding

In [97]:
torch.set_float32_matmul_precision('medium')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

peptides_file = '../data/processed/c_train_113559samples_10pct_per_class.csv'
pretrain_model = 'facebook/esm2_t33_650M_UR50D'

In [98]:
print(f"Loading data from: {peptides_file}")
peptides_dataset = PeptideDataset(file_path=peptides_file, seq_col='cleavage_window', label_col='elution_cleavage')


Loading data from: ../data/processed/c_train_113559samples_10pct_per_class.csv


In [99]:
print("Initializing embedder...")
embedder = Embedder(name=pretrain_model, device=device)
print("Generating embeddings for all windows...")
# This step can consume a lot of memory for large files
seq_embeddings = embedder.encode_esm(peptides_dataset, emb_type='concat', batch_size=64, num_workers=8)


Initializing embedder...
Initializing protein sequence embedder ... 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Generating embeddings for all windows...


Generating Embeddings: 100%|███████████████████████████████████████████████████| 113559/113559 [02:43<00:00, 695.05it/s]


In [112]:
seq_embeddings.shape,peptides_dataset.data[peptides_dataset.label_col].values.shape

(torch.Size([113559, 1, 12800]), (113559,))

In [108]:
seq_embeddings.cpu().numpy(),peptides_dataset.data[peptides_dataset.label_col].values

tensor([[[-0.0631,  0.1115, -0.0318,  ..., -0.2406,  0.1374, -0.0401]],

        [[ 0.0060,  0.1029, -0.0172,  ..., -0.0182, -0.1476,  0.1477]],

        [[-0.0257, -0.0088,  0.0116,  ..., -0.1152, -0.0784,  0.0271]],

        ...,

        [[ 0.1437,  0.0233,  0.0326,  ..., -0.0506, -0.1468, -0.0183]],

        [[ 0.0636,  0.0557,  0.0927,  ..., -0.0479, -0.2346, -0.0232]],

        [[-0.0021,  0.0673, -0.0565,  ..., -0.0801, -0.1312, -0.0375]]])

In [109]:
import h5py

In [106]:
peptides_dataset.data[peptides_dataset.label_col].values

array([0, 0, 1, ..., 0, 1, 1])

In [ ]:
def save_embeddings(emb_dict, out_path):
    with h5py.File(str(out_path), "w") as hf:
        for sequence_id, embedding in emb_dict.items():
            # noinspection PyUnboundLocalVariable
            hf.create_dataset(sequence_id, data=embedding)
    return None